In [3]:
import numpy as np
from collections import Counter
from math import log

# Function to read documents from a file
def read_documents(doc_file_path):
    documents = []
    labels = []
    with open(doc_file_path, encoding='utf-8') as file:
        for line in file:
            words = line.strip().split()
            if len(words) < 4:
                continue  # Skip lines with insufficient content
            labels.append(words[1])
            documents.append(words[3:])
    return documents, labels

# Function to train the Naive Bayes classifier
def train_naive_bayes(documents, labels, smoothing=1):
    positive_word_counts = Counter()
    negative_word_counts = Counter()
    total_positive_words = 0
    total_negative_words = 0

    for document, label in zip(documents, labels):
        if label == 'pos':
            positive_word_counts.update(document)
            total_positive_words += len(document)
        else:
            negative_word_counts.update(document)
            total_negative_words += len(document)

    vocabulary = set(positive_word_counts.keys()) | set(negative_word_counts.keys())
    positive_probabilities = {word: (positive_word_counts[word] + smoothing) / (total_positive_words + smoothing * len(vocabulary)) for word in vocabulary}
    negative_probabilities = {word: (negative_word_counts[word] + smoothing) / (total_negative_words + smoothing * len(vocabulary)) for word in vocabulary}
    return positive_probabilities, negative_probabilities

# Function to score a document against label probabilities
def score_document_label(document, label_probabilities):
    log_probability = 0
    for word in document:
        if word in label_probabilities:
            log_probability += log(label_probabilities[word])
    return log_probability

# Function to classify a document as positive or negative
def classify_naive_bayes(document, positive_probabilities, negative_probabilities):
    positive_score = score_document_label(document, positive_probabilities)
    negative_score = score_document_label(document, negative_probabilities)
    return 'pos' if positive_score > negative_score else 'neg'

# Function to classify a collection of documents
def classify_documents(docs, positive_probabilities, negative_probabilities):
    return [classify_naive_bayes(doc, positive_probabilities, negative_probabilities) for doc in docs]

# Function to calculate accuracy
def calculate_accuracy(true_labels, predicted_labels):
    if len(true_labels) == 0:
        return 0  # Return 0 if true_labels is empty to avoid ZeroDivisionError
    correct_predictions = sum(1 for true, predicted in zip(true_labels, predicted_labels) if true == predicted)
    return correct_predictions / len(true_labels)

# Load and process data
all_documents, all_labels = read_documents('all_sentiment_shuffled.txt')
split_point = int(0.80 * len(all_documents))
training_documents = all_documents[:split_point]
training_labels = all_labels[:split_point]
evaluation_documents = all_documents[split_point:]
evaluation_labels = all_labels[split_point:]

# Train the classifier and evaluate on test set
positive_probabilities, negative_probabilities = train_naive_bayes(training_documents, training_labels)
predicted_labels = classify_documents(evaluation_documents, positive_probabilities, negative_probabilities)
accuracy = calculate_accuracy(evaluation_labels, predicted_labels)
print("Accuracy on the test set:", accuracy)

# Cross-validation
NUM_FOLDS = 5  # Number of folds for cross-validation
accuracies = []

# Perform cross-validation
for fold_number in range(NUM_FOLDS):
    split_point_1 = int(float(fold_number) / NUM_FOLDS * len(all_documents))
    split_point_2 = int(float(fold_number + 1) / NUM_FOLDS * len(all_documents))
    train_documents_fold = all_documents[:split_point_1] + all_documents[split_point_2:]
    train_labels_fold = all_labels[:split_point_1] + all_labels[split_point_2:]
    eval_documents_fold = all_documents[split_point_1:split_point_2]
    eval_labels_fold = all_labels[split_point_1:split_point_2]
    positive_probabilities_fold, negative_probabilities_fold = train_naive_bayes(train_documents_fold, train_labels_fold)
    predicted_labels_fold = classify_documents(eval_documents_fold, positive_probabilities_fold, negative_probabilities_fold)
    accuracy_fold = calculate_accuracy(eval_labels_fold, predicted_labels_fold)
    accuracies.append(accuracy_fold)

# Compute mean and standard deviation of accuracies
mean_accuracy = sum(accuracies) / NUM_FOLDS
standard_deviation_accuracy = np.std(accuracies)

print("Mean accuracy with cross-validation:", mean_accuracy)
print("Standard deviation of accuracy with cross-validation:", standard_deviation_accuracy)

Accuracy on the test set: 0.8136802349979018
Mean accuracy with cross-validation: 0.809719419636644
Standard deviation of accuracy with cross-validation: 0.003070995465360702
